# Group-aware DDR split
DDR has no patient IDs, so the random 60/20/20 split can put the same eye (or the same hospital's camera) in both train and test.
This notebook builds two safer splits and measures how much the random split leaked. **Nothing is retrained unless you switch it on in the last code cell, and no test set is opened.**

**Before running**
* Accelerator **GPU**, Internet **On** (it downloads ImageNet weights to compare images).
* Add as inputs: (1) your **old notebook's output** (the one with the finished runs), (2) the **code dataset** (`Early-DR` zip), (3) the **DDR dataset** (needed for the camera-size split; the group split works without it).

**What you get**
* `audit/ddr_split_report.csv`: split sizes, grade mix, near-duplicate leakage and image-property AUROC for the random, group and camera splits.
* `figures/ddr_similarity_pairs.jpg`: pairs at different similarity levels, so you can see where "same eye" starts.
* `v2/`: a new work folder (splits, manifest, linked images) that uses the split you choose. Your old results stay untouched.

In [ ]:
# 1. Set-up: code folder, the grouping module, and the finished run from the old notebook's output
import base64, glob, json, os, shutil, subprocess, sys, zlib
INPUT = os.environ.get("KAGGLE_INPUT", "/kaggle/input")
WORK = os.environ.get("EARLYDR_WORK", "/kaggle/working")
CODE = os.environ.get("EARLYDR_CODE", os.path.join(WORK, "code"))
PRETRAINED = os.environ.get("EARLYDR_NOPRETRAIN") is None
MODULE = (
    "eNqtWm1v20YS/u5fsacCV7KRGUmJ28SIepcmThqglwS2gbuDIKgrciVtRJEsd2lLCfLf75nZ5Zts5wrcGYFEkbvzPs/MLDMYDN6W"
    "eVWcyltZKmGKVFsjVnkpXr++FIG2YiONyHJRSKtVZsW712YoTC7sRolSZkm+Ez+OHk/on9stYpmJVMltGJ2cXN/mQqZWlRm23ygj"
    "bG8nbzg/EWJNMggh9E6uadVGWpHm+Vakeqt4h5E7JdRBiSBTsjxNKmyNpVUmBINbeTAiBU2hs3Y1UwfxGD9KKcTtJk+VYyGM/gw+"
    "gRRFme8PrDDt2+Sm0Fam4rHfBeowy0alicgr22gOvlYYBWXzzEqdGbCp/2JZ2KpkLZTY5YlKRaZuVAmhbkVSlTpbC1tiEy5OTi7w"
    "5GA3dHOjwKoytJMeP76RqU5gtzw71VmiCoUPeEBnkHbH90WepQd4yVmN9Ge9huQwxxlyVKk18MVgMDjRuyIvrVin+bK+zs1JfRnf"
    "TOrLrNoVB0GeL+pbBcjjBv4VSX3P5mW8OVmVcOXHd78Jf/cdiePumi0iocwilRm1W7L1ecmv2ti3pUwopH7Jc2Oh/6tUGqNXWpX9"
    "vTtlSx2bemuZxwtZxQsT56U6cUsjmEmSO2pN8jSvykVGdkq1wbKTRK3I17EyRiWLlU5VcJuX26GLh4VOwnP24SpPE/hqCk+XViUB"
    "2SqijyA3EZJgE33Kdeb3DlqKPwzCMJyN5kykVIgAeKe7wxFu+YlHYhB9KtaDENJ9J07pT/Rj26WFcc/+xz9nAThBJQmMbYIkgTBO"
    "jUTd6FgNOXimk8nToVhKG2+m48mzIYymOBxVMr0uK+XNhGj6kCFaM21PU5Wt7UbcqBjxIApYj3U8d4HwHm5ZKelSIl9xVjj/nDb+"
    "SUShY1oBiTYaIUUs1K4A9mjjM+mG7JwSDhGciKAHCmsGF5nuEEotADiJwohCXzC6uKjVux3/diky5RtRXEJIteB7wUCtVjqm6MyU"
    "XSxHg54d2sshZcoipshVZjoKI5sHzpphpJC/QegYKWDi1KVLZJEMeRnMRtHTZ2dDga+zH/lr9OO89sW0JnKj1W0wHoonQ4HPsSNn"
    "bHIPtcnkOZFh/9HX2Z+kBlwDtZmLXJ0Y/IBPLAVI1CSHzw2AmJUlYRBB+FoFoyGwPguwLfRR4wNEtGjeEK9pNCkAMtg5Y5rnjvIj"
    "R2V+3kFUorQGGQBUFN/YV4ieMqAfegenJcG38xqC0dpXH377cLn45e3l5PLtL+ED1BGjyIHgGD+AsGuQCeihSxP80hkCschTRuIp"
    "7X73/vricvHy8uLlMX2yQyQLgnCm1TzeHzsyKyIYIt56UA/v+BD5tassW965cRJGqzSXNghRtSZnZ9GooX6rkZeOfJYv1gDcIOwb"
    "Fs6v5fLrsmhVZTHpJNPIWwAmcXkR7MUpRzPxQhiSeHo3HYdRXFRBGHbRz9EDjgVgEnoIzpReb5YwrgmARS30bKdjKBRvqmw7nYye"
    "Pmth5g0ChsrnwZkRdkeDshWc6UbvdCpL8eH614tLb+Zz1MMsAVGUwTg3yNJ6mUYHY1o0AH8YH5+dpPU5sCcn7zhwh3XwPhz8IBF6"
    "2TvWNY74UXDzqrn4O/O9bhfPnLWkI2wis5GFQjkB4f4TojMUNbnOwjn4nY4due/QAFgylEpXDQ+bF1TWoG2xDba149pkTfZtJBSR"
    "t2HXryyo3nVXAeKq/qI7zgfdMOJ+ImiUofug1Nz3wSErmy/sBkmITi1xFYqjRFaJtovY3LRh8VHHW4f0tXPRryzzG+UqiLDoPD3+"
    "xHmFtkmaXh8ZiWu5VZng/qGQunSPUXE2jh0zkikBDPWHFZorTidNTRjKcyoon6iJ5OCkRhW9mEWJFJ3m9Jyo6lKc2c0pMjemvWlX"
    "6LY4rdhpdc+g9sBgExxr3jHxKHp+xvcS6hGmaMsiEpYWd7Z1V9DXjD4igwyMlZhOxQB9/sB3LfktFn2pgfNcaIfWwx5gK/gMjTEQ"
    "qFcgvp748OjgPe1eDMWatjFb7meWh2DAF4OOTgTA69m6oRdpZG4AicK2dMBClG7rUPw8FRNud9cROwFxlOk/KgUIhErjPsSt4Fuq"
    "cCDWY5DmMfJm3lvL8j+CAg5RKX95u8/YGdHQ83noDEN6HVMcn3uSXlqiGIoXYjx62IH+t+OJAhCnuqDvNmCYzFCchVTanz2jz+fP"
    "z8I6cTh+Fxz+QY1eyLU6k7ydt0oxAgDZYMDmae17chyYlhg0nBgeclx931MdJYxwgDOeh1Qs6Tpo7rpgk2gEnLnNjFjOeZ+77Krr"
    "G+wvwQ6eRpB9AoOd3LvLsA69T2Tkz7AHkQ2/1hovK50mC9cfB9lQ3KM1U1tAsFJNR9Fo0mLHqzzL0Bqi7fQNdr466rxNJN6tANZu"
    "SgOhLTWvGTWaEljixlXUxJKGU/R2DSciJdP0pC37BHo0MdZiCW1QPrjphWYG5TUFnCtIUZGrgdo0vaU5TF+KJCdRLtlgQBrHFy1T"
    "R02aGJO2quG+OHYuEBGYQ917J+Gc5lOaYJ0xOtZ8MJ46lQDwtVacWACeK1Witjr5UGK5LCwYd00Q1qnWS2S/+8W0Y7sfBGamklVA"
    "gHKI91PZR45jwwJxGxZ4aq1wzgoloXZA149AbTRCt/2kDqAH1fZhUkCgzNa9sMuEjBKO8RQEVqiQwb4DYc7Kbt9sPxd/mYp9X/z2"
    "2bS+bm71cWjfrtjPj3Fj34ArpxqiiAVvefmdLKHkxoAvl70KzQleykPg1ukOptXazmtj7eiUY626M2MbFkudmWkQNLj0hDpl+uZf"
    "P/lfP6FXjUZjQhHA2gKbpk/9xDmeoJEyCoPVqFPfuSRLizZltVLsi06lT9ERpnQQhWoslgf+5sOZQ17xARTIwSF0oPJ9XfG/d12T"
    "aVKlzNYe8vg0KrrkrytLtY3ECbvI2ClpaT4UGzYUqd4ZdxYeQplxQJg9O4cR5hTNaR6Kv4rOvReg0Z4YdCoGyHAZG/WDh06adFap"
    "1svogIgh1IjiTY5uLai3A/qAqd7Ow4Yq7gPdU4lp4o1MjerkC0L3nkGt4OAiPkfjGOHylBSe1fBeX4/mRyGvyz7dmvaWaDuw7xPv"
    "T2TfGvG6DYgDme08DO8QIxnqprUz5IFHf6Q72kpLi8peq70NiARcNRSrwRdyoSZFz6Mnq68D0KA57KkfM998eH+9wDRy9evFvxdX"
    "7/7x8beLf7khn/JgSAMaf4TN/N3zQS0ndQF5RqUoo3Ak/miD0ROiZQ+PIIfcy5upz/BOP/8m6c+qzM2CWla3keaHlipF/ANy8Ooj"
    "QVo86a4kIn7hKKTQ5jxC0irxHoW0c+blz5xP//9/fqowRq+zGub5a8EldFVKnnONQ58We/7JB8W+Qq5zOrSm2s+CRuJCYrzwldgI"
    "TqfEYw0PGHwIjierqsRv1LilStFZk46W6pRtACgDMnHWDfg4CaE0QOWkLzpfrrvyP4VSrv91C30v3Gga1nTQpVWrFdLHrfCHSXwS"
    "fqeK88ajQu4aDNbBcXIFpLFjiArOHXqfMzI2VYnbwZEXPAl9X0JGw8jxtcFWnhWceG0IbwlrMt+TrgnaPMnHXppeSoDmbE1Fj80L"
    "SGghjXfhDrX4rDcW3l8TGzKNUI1STVV00MN0qDL6+Knt2SkYNNDxDAS8outWM1se+pnKIyafm0Y5EjCg5SHNrnp3FyWZVXusFDGK"
    "NavUPlaFFR+uLsoyL89PHtxJsDTqJzM/92rSu5ym+HN039tut+lzVTfCiJ7H/KbED+KsnDw+5A46db07l4T1y5zry5fv3vvsq1PH"
    "ZZkLQuN8xre6R5qdsKJQIHKzfoKd9zCvW7w9fRdEvSq9Yb4zt2JGpa+cz3mSdkkczqhAlPOeLvNIZofAtVglt1jg19KEvDNiNG+a"
    "1mYWpMO2ADypeD8N6x6BtocOTbEmk1nvtUNl675NWUmvRhayQv3seZA93BmJ0DPd6gT+eSw2dEZncbFyRxWfFUYaAsCiVImOrYD/"
    "0oOg94P+CGR8OgnFjQGy093cuY1emp2SJ9zLMmL7t2ZKkfQyiEHnNeR7U0L54MvgdnAuZlSN3GkbD0gkJwrvYOOeje8+u5MX7d9g"
    "u6Rt9aEKoIJLf0HHl+PR5CnZ8+jEpfB2HbUdUJ2332bFtgA3Wu1OJdzpGGRn5fHoTrR+DXvm4O8ZfzoSNB9NXJxwdA2dMfsrm1j1"
    "EQhrPfC4rSrQm355Qdc2GIWdQ5QXYnLntMJF5ACxNgh7b0++/S4voPFOW1VOx2fAGVfEFobKF70vWWkbsNSzGbzPbh6y1+akBT3w"
    "hvgZENUN8m6a9F4JBq1etGno5Ix87C7QSC4lr7nLMaTWfDynVGvmxKVMZRZ30c8njZ98Y9QzaLMM7vh2eE8kNOfp0wG90t0Pwsgp"
    "8oRq53jkju5thLlClZaAeeCwk15ANQX6DqvjQt0/ikXDy8yCh/qMWtdbYLBa3ACKJartTmZ6heesR4MbSOhFQs1opyPnN5stlrwE"
    "wt/ymFi/TeX3+fxm+3cm83v7PwwuDurjy1dXfJr38uP1h6u6H6yyeENzaOdsA5lKJzFgb4JGDk7bRb51byddX6XoILOWH20pnzH7"
    "JxHNCfR9dAbapOm81wzd41Y25pSHD7pqKducD197r30bOR11E2HFoCHSGcP+O4FWoQdIpDqjVun+7fXdpTSKak1wx4HhN86hiXRn"
    "TsNTc9jRzTtkhixGfzRQtyf/AYLOqjE="
)
if not os.path.exists(os.path.join(CODE, "early_dr", "dataset.py")):
    found = glob.glob(os.path.join(INPUT, "**", "steps", "settings.py"), recursive=True)
    if not found:
        sys.exit("STOP: add the code dataset (the Early-DR zip you uploaded) as an input of this notebook.")
    shutil.copytree(os.path.dirname(os.path.dirname(found[0])), CODE, dirs_exist_ok=True)
try:
    import timm  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm"])
open(os.path.join(CODE, "early_dr", "regroup_ddr.py"), "w").write(zlib.decompress(base64.b64decode(MODULE)).decode())

if not os.path.exists(os.path.join(WORK, "manifest.csv")):
    candidates = [os.path.dirname(f) for f in glob.glob(os.path.join(INPUT, "**", "splits.csv"), recursive=True)]
    candidates = [c for c in candidates if os.path.abspath(c) != os.path.abspath(WORK)]
    if not candidates:
        sys.exit("STOP: add your old notebook's output as an input of this notebook.")
    previous = candidates[0]
    print("restoring from", previous)
    for name in ["splits.csv", "manifest.csv"]:
        shutil.copy(os.path.join(previous, name), os.path.join(WORK, name))
    for name in ["audit", "figures"]:
        if os.path.exists(os.path.join(previous, name)):
            shutil.copytree(os.path.join(previous, name), os.path.join(WORK, name), dirs_exist_ok=True)
    for name in os.listdir(previous):
        if name.startswith("processed_") and not os.path.exists(os.path.join(WORK, name)):
            os.symlink(os.path.join(previous, name), os.path.join(WORK, name))
os.makedirs(os.path.join(WORK, "figures"), exist_ok=True)
sys.path.insert(0, CODE)
print("ready")

## 2. Compare DDR images and find near-duplicates
Each image becomes a vector (vessel pattern); images of the same eye end up almost identical. The similarity cut-off is chosen from pairs your earlier hash audit already found with identical grades.

In [ ]:
import numpy as np, pandas as pd, torch
from early_dr import regroup_ddr as R

manifest = pd.read_csv(os.path.join(WORK, "manifest.csv"))
ddr = manifest[manifest.source == "DDR"].reset_index(drop=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(len(ddr), "DDR images on", device)
emb = R.embeddings(ddr, WORK, device, pretrained=PRETRAINED)
idx, sim = R.neighbours(emb, device)
THRESH_AUTO = R.auto_threshold(ddr, emb, os.path.join(WORK, "audit", "duplicates.csv"))
print("most similar other image, per image: median %.3f, 95th percentile %.3f" % (np.median(sim[:, 0]), np.percentile(sim[:, 0], 95)))
print("suggested similarity cut-off for 'same eye':", round(THRESH_AUTO, 3))

## 3. Look at pairs, then set the cut-off
Each row is a similarity band (low to high). Pairs in the top rows should look like the SAME eye; if the lower band still shows the same eye, lower the cut-off; if the top band shows different eyes, raise it. Leave `THRESH = None` to use the suggestion.

In [ ]:
import cv2
from IPython.display import Image, display
picture = R.montage(ddr, WORK, idx, sim)
if picture is not None:
    cv2.imwrite(os.path.join(WORK, "figures", "ddr_similarity_pairs.jpg"), picture)
    display(Image(os.path.join(WORK, "figures", "ddr_similarity_pairs.jpg")))
THRESH = None          # e.g. 0.96 to override the suggestion

## 4. Build the splits and measure the old one
* **random**: the split you used so far.
* **group**: near-duplicate groups stay together (60/20/20).
* **camera**: whole image sizes (a proxy for hospital / camera) are held out.

`leak` is the share of validation/test images that have a near-duplicate in train. `size AUROC` is how well image width, height and file size alone predict early DR on the test split (0.5 = no leak of the camera).

In [ ]:
SEED, FRACTIONS = 2024, (0.6, 0.2, 0.2)
thr = THRESH or THRESH_AUTO
groups, thr_used, largest = R.build_groups(len(ddr), idx, sim, thr)
print(f"cut-off {thr_used}: {len(set(groups))} groups, largest group {largest} images")

variants = {"random": ddr.split.values, "group": R.assign_groups(groups, FRACTIONS, SEED)}
sizes = R.image_sizes(ddr)
n_sizes = len(set(sizes))
if n_sizes >= 4 and set(sizes) != {(0, 0)}:
    size_ids = pd.factorize(pd.Series([f"{w}x{h}" for w, h in sizes]))[0]
    variants["camera"] = R.assign_groups(size_ids, FRACTIONS, SEED)
else:
    print("camera split skipped: add the DDR dataset as an input (image sizes could not be read)")

rows = []
for name, split in variants.items():
    leak = R.leakage(ddr, split, idx, sim, thr_used)
    counts = pd.Series(split).value_counts()
    rows.append({"split": name, "train": counts.get("train", 0), "val": counts.get("val", 0), "test": counts.get("test", 0),
                 "leak val": leak["val"], "leak test": leak["test"], "size AUROC": R.metadata_auroc(ddr, split, sizes)})
report = pd.DataFrame(rows)
print(report.to_string(index=False))
for name, split in variants.items():
    print(f"\ngrade mix (% of each split), {name}"); print(R.balance(ddr, split).to_string())
os.makedirs(os.path.join(WORK, "audit"), exist_ok=True)
report.to_csv(os.path.join(WORK, "audit", "ddr_split_report.csv"), index=False)
pd.DataFrame({"image_id": ddr.image_id, **{k: v for k, v in variants.items()}}).to_csv(os.path.join(WORK, "audit", "ddr_splits_all.csv"), index=False)

## 5. Write the new work folder
Pick the split to use. `group` fixes near-duplicate leakage; `camera` is stricter because the test set uses image sizes never seen in training (expect lower scores). Your old results are not touched.

In [ ]:
CHOICE = "group"        # "group" or "camera"
V2 = os.path.join(WORK, "v2")
processed = sorted(glob.glob(os.path.join(WORK, "processed_*")))[0]
new = R.write_variant(manifest, ddr, variants[CHOICE], V2, processed)
print(f"wrote {V2} using the '{CHOICE}' DDR split")
print(new.groupby(["source", "split"]).size().unstack(fill_value=0))

## 6. (Optional) retrain on the new split
Off by default. Turning it on trains arms A, B and C with **one seed** on the new split (roughly 2 hours on a GPU); results go to `v2/runs`. Evaluating on the test sets is a separate step (steps 6 and 8 with `paths.work=/kaggle/working/v2`) and counts as a new, declared analysis.

In [ ]:
RUN_TRAINING = False
if RUN_TRAINING:
    for arm in ["arm_A", "arm_B", "arm_C"]:
        subprocess.run([sys.executable, "-m", "early_dr.run_experiment", f"configs/arms/{arm}.yaml", "--set",
                        "task=stage1", f"paths.work={V2}", "--stage", "train", "--seeds", "42"], cwd=CODE)
else:
    print("training is off: set RUN_TRAINING = True to retrain on the new split")

## What to send me
`audit/ddr_split_report.csv` (or the printed table from step 4) and `figures/ddr_similarity_pairs.jpg`. The table shows how much the random split leaked and whether image properties still predict the label on each split.